# Gain per variation, per benchmark — measured

Every variation we add (device, slice width, sweeps, stopping accuracy, live points, …) is a
parameter whose default is today's behaviour. `sweep.py` runs a grid of them on a benchmark
and records, per run, the evidence and its uncertainty, the likelihood calls, the wall time,
the insertion-index check, and the error-evolution trace. This notebook reads those files and
answers one question per benchmark: **what does each variation buy, and does it move the
evidence?**

Gain is *relative to the defaults*, in likelihood calls (the hardware-independent cost) and in
wall time. A variation only counts if the evidence stays within the reference uncertainty.

In [1]:
import json, pathlib, re
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # embeds plotly.js: renders in any JupyterLab, no extension needed
from plotly.subplots import make_subplots

BLUE, ORANGE, GREY, INK, SURFACE, GRID = "#2a78d6", "#eb6834", "#6b6b68", "#1a1a19", "#fcfcfb", "#e6e6e3"
SWEEPS = pathlib.Path.home() / ".cache/maqi-research/nested_fit/sweeps"      # where sweep.py outputs are dropped
if not SWEEPS.exists(): SWEEPS = pathlib.Path(".")
DEFAULTS = {"device": "cpu", "param1": "0.5", "param2": "3", "accuracy": "1e-5", "livepoints": None, "method": "SLICE_SAMPLING"}
REFERENCE = {"test_12_gpu_crosscheck": (-951.25, 0.12), "test_01_gauss_bg": (-574.5, 1.0), "test_03_4gauss_bg": (-326.1, 0.5)}
FIXED = {"case","repeat","exit","wall_s","logZ","uncert","maxlnL","calls","iterations","insertion_ks_p","insertion_ks_p_first","gpu"}

def load(path):
    df = pd.read_csv(path, sep="\t", dtype=str)
    for c in ("logZ","uncert","calls","wall_s","iterations"): df[c] = pd.to_numeric(df[c], errors="coerce")
    df["params"] = [c for c in df.columns if c not in FIXED] and [tuple(c for c in df.columns if c not in FIXED)]*len(df)
    return df

def style(fig, title, h=420):
    fig.update_layout(title=title, height=h, plot_bgcolor=SURFACE, paper_bgcolor=SURFACE, font=dict(color=INK),
                      margin=dict(t=60, b=40, l=10, r=10), legend_font_color=INK)
    fig.update_xaxes(gridcolor=GRID); fig.update_yaxes(gridcolor=GRID); return fig

sweeps = {p.stem: load(p) for p in sorted(SWEEPS.glob("*.tsv"))}
print("sweeps found:", list(sweeps) or "none — run sweep.py first")

sweeps found: ['sweep_test12_v3', 'sweep_test12_width_stop', 'sweep_width_t01', 'sweep_width_t12']


## 1. Gain per variation

For each parameter varied in a sweep, the runs where *only that parameter* differs from the
defaults. Bars are the mean gain over repeats (> 1 = better); the bar is blue when the
evidence agrees with the reference within its uncertainty, orange when it does not.

In [2]:
def baseline_mask(df, params):
    m = pd.Series(True, index=df.index)
    for p in params:
        d = DEFAULTS.get(p)
        if d is not None: m &= df[p].astype(str).str.lower() == str(d).lower()
    return m

def gains(df):
    params = [c for c in df.columns if c not in FIXED and c != "params"]
    case = df["case"].iloc[0]; ref, ref_unc = REFERENCE.get(case, (df["logZ"].mean(), df["logZ"].std() or 1))
    base = df[baseline_mask(df, params)]
    if base.empty: base = df.iloc[:max(1, len(df)//len(df.drop_duplicates(params)))]
    b_calls, b_wall = base["calls"].mean(), base["wall_s"].mean()
    rows = []
    for p in params:
        others = [q for q in params if q != p]
        sub = df[baseline_mask(df, others)] if others else df
        for val, g in sub.groupby(p):
            rows.append(dict(case=case, parameter=p, value=val, runs=len(g), calls_gain=b_calls/g["calls"].mean(),
                             wall_gain=b_wall/g["wall_s"].mean(), dlogZ=g["logZ"].mean()-ref,
                             z=abs(g["logZ"].mean()-ref)/np.hypot(ref_unc, g["uncert"].mean()),
                             is_default=str(val).lower()==str(DEFAULTS.get(p,"")).lower()))
    return pd.DataFrame(rows)

for name, df in sweeps.items():
    G = gains(df); case = df["case"].iloc[0]
    params = list(dict.fromkeys(G["parameter"]))
    fig = make_subplots(rows=1, cols=len(params), subplot_titles=[f"{p}" for p in params], shared_yaxes=True)
    for i, p in enumerate(params, 1):
        g = G[G.parameter==p]
        fig.add_trace(go.Bar(x=[str(v) for v in g.value], y=g.calls_gain, name="calls gain", showlegend=(i==1),
                             marker_color=[BLUE if z<1 else ORANGE for z in g.z], text=[f"{v:.2f}×" for v in g.calls_gain], textposition="outside",
                             customdata=np.c_[g.dlogZ, g.z, g.wall_gain], hovertemplate=p+"=%{x}<br>calls gain %{y:.2f}×<br>wall gain %{customdata[2]:.2f}×<br>Δ log Z %{customdata[0]:+.4f} (%{customdata[1]:.1f} σ)<extra></extra>"), row=1, col=i)
        fig.add_hline(y=1, line_dash="dash", line_color=GREY, row=1, col=i)
    fig.update_yaxes(title_text="gain in likelihood calls vs defaults", row=1, col=1)
    style(fig, f"{case}: gain per variation ({name}) — blue: evidence within reference uncertainty, orange: not")
    fig.show()
    display(G.round(4))

,case,parameter,value,runs,calls_gain,wall_gain,dlogZ,z,is_default
0,test_12_gpu_crosscheck,device,cpu,2,1.0000,1.0000,-0.1144,0.6744,True
1,test_12_gpu_crosscheck,param1,0.5,2,1.0000,1.0000,-0.1144,0.6744,True
2,test_12_gpu_crosscheck,param1,1,2,1.4791,1.4338,0.0197,0.1160,False
3,test_12_gpu_crosscheck,param1,2,2,1.8947,1.6882,-0.2038,1.2006,False
4,test_12_gpu_crosscheck,accuracy,1e-3,2,1.1303,1.2054,0.0772,0.4559,False
5,test_12_gpu_crosscheck,accuracy,1e-5,2,1.0000,1.0000,-0.1144,0.6744,True


,case,parameter,value,runs,calls_gain,wall_gain,dlogZ,z,is_default
0,test_12_gpu_crosscheck,device,cpu,2,1.0000,1.0000,0.0922,0.5444,True
1,test_12_gpu_crosscheck,param1,0.5,2,1.0000,1.0000,0.0922,0.5444,True
2,test_12_gpu_crosscheck,param1,1,2,1.4693,1.3264,-0.0736,0.4344,False
3,test_12_gpu_crosscheck,param1,2,2,1.8844,2.4295,-0.2698,1.5887,False
4,test_12_gpu_crosscheck,accuracy,1e-3,2,1.1222,1.1193,-0.0983,0.5797,False
5,test_12_gpu_crosscheck,accuracy,1e-5,2,1.0000,1.0000,0.0922,0.5444,True


,case,parameter,value,runs,calls_gain,wall_gain,dlogZ,z,is_default
0,test_01_gauss_bg,device,cpu,2,1.0000,1.0000,-0.3389,0.3388,True
1,test_01_gauss_bg,param1,0.5,2,1.0000,1.0000,-0.3389,0.3388,True
2,test_01_gauss_bg,param1,1,2,1.4741,1.2940,-0.3580,0.3578,False
3,test_01_gauss_bg,param1,2,2,1.9012,1.5264,-0.3662,0.3660,False


,case,parameter,value,runs,calls_gain,wall_gain,dlogZ,z,is_default
0,test_12_gpu_crosscheck,param1,1,4,1.0000,1.0000,-0.0781,0.4605,False
1,test_12_gpu_crosscheck,param1,1.5,4,1.1776,1.1274,-0.1478,0.8714,False
2,test_12_gpu_crosscheck,param1,2,4,1.2863,1.1260,-0.1250,0.7367,False


## 2. Error evolution — do we run longer than the evidence needs?

The remaining-evidence estimate `Ea` against the step, one line per variation. Where a line
crosses 10⁻³ it has already converged to within a thousandth in log Z — a hundred times below
the reference uncertainty. Everything to the right of that crossing is work spent on precision
that cannot be seen.

In [3]:
for name in sweeps:
    pdir = SWEEPS / f"{name}.progress"
    traces = sorted(pdir.glob("*.json")) if pdir.exists() else []
    if not traces: print(name, ": no progress traces (older sweep)"); continue
    fig = go.Figure(); seen = {}
    for t in traces:
        d = json.loads(t.read_text()); tag = ", ".join(f"{k}={v}" for k, v in d["params"].items())
        if d["repeat"] != 1 or not d["progress"]: continue          # one line per variation
        s, z, ea = zip(*d["progress"]); col = [BLUE, ORANGE, GREY, INK][len(seen) % 4]; seen[tag] = 1
        fig.add_trace(go.Scatter(x=s, y=ea, name=tag, line=dict(width=2, color=col, dash=["solid","dash","dot","dashdot"][len(seen)//4 % 4])))
    for thr, pos in ((1e-3, "top right"), (1e-5, "bottom right")): fig.add_hline(y=thr, line_dash="dash", line_color=GREY, annotation_text=f"Ea = {thr:g}", annotation_position=pos)
    fig.update_yaxes(type="log", title="remaining-evidence estimate Ea"); fig.update_xaxes(title="step")
    style(fig, f"{name}: error evolution"); fig.show()

## 3. Everything on one axis

Evidence error against likelihood calls, all runs. Lower-left is better; points under the
dashed line are indistinguishable from the reference, and among those fewer calls wins.

In [4]:
for name, df in sweeps.items():
    case = df["case"].iloc[0]; ref, ref_unc = REFERENCE.get(case, (df["logZ"].mean(), 0))
    params = [c for c in df.columns if c not in FIXED and c != "params"]
    key = params[0] if params else None
    fig = go.Figure()
    for i, (val, g) in enumerate(df.groupby(key) if key else [("runs", df)]):
        fig.add_trace(go.Scatter(x=g.calls, y=(g.logZ-ref).abs(), mode="markers", name=f"{key}={val}" if key else "runs",
                                 marker=dict(size=10, color=[BLUE, ORANGE, GREY, INK][i % 4]),
                                 customdata=np.c_[g.logZ, g.uncert, g.wall_s], hovertemplate="calls %{x:,.0f}<br>|Δ log Z| %{y:.4f}<br>log Z %{customdata[0]:.4f} ± %{customdata[1]:.4f}<br>wall %{customdata[2]:.0f} s<extra></extra>"))
    if ref_unc: fig.add_hline(y=ref_unc, line_dash="dash", line_color=GREY, annotation_text="reference uncertainty")
    fig.update_xaxes(title="likelihood calls"); fig.update_yaxes(title="|log Z − reference|", rangemode="tozero")
    style(fig, f"{case}: evidence error vs cost ({name})"); fig.show()
    if "insertion_ks_p" in df and df["insertion_ks_p"].notna().any():
        display(df[params + ["repeat","logZ","uncert","calls","wall_s","insertion_ks_p"]])

,device,param1,accuracy,repeat,logZ,uncert,calls,wall_s,insertion_ks_p
0,cpu,0.5,1e-5,1,-951.510882,0.120073,14880917,31.6,0.00817
1,cpu,0.5,1e-5,2,-951.217841,0.119574,14786218,31.2,0.0288
2,cpu,0.5,1e-3,1,-951.108018,0.119298,13096595,26.1,0.0218
3,cpu,0.5,1e-3,2,-951.237618,0.119568,13149475,26.0,0.00403
4,cpu,1,1e-5,1,-951.111762,0.119381,10002148,21.8,0.000568
5,cpu,1,1e-5,2,-951.348938,0.119937,10055934,22.0,0.00658
6,cpu,1,1e-3,1,-951.248989,0.119572,8896807,19.7,0.0152
7,cpu,1,1e-3,2,-951.396950,0.119857,8930961,19.8,0.00181
8,cpu,2,1e-5,1,-951.381145,0.119909,7821262,18.5,0.00366
9,cpu,2,1e-5,2,-951.526434,0.120196,7837011,18.7,0.0119


,device,param1,repeat,logZ,uncert,calls,wall_s,insertion_ks_p
0,cpu,0.5,1,-574.829773,0.029631,25237415,79.1,0.0369
1,cpu,0.5,2,-574.848032,0.029634,25268091,79.8,0.0942
2,cpu,1,1,-574.833601,0.029636,17117382,61.5,0.0617
3,cpu,1,2,-574.882347,0.029691,17145086,61.3,0.0942
4,cpu,2,1,-574.886207,0.029725,13292694,51.9,0.0496
5,cpu,2,2,-574.846200,0.029647,13272412,52.2,0.00767


,device,param1,repeat,logZ,uncert,calls,wall_s,insertion_ks_p
0,cpu,1,1,-951.225947,0.119678,10041752,22.0,0.0094
1,cpu,1,2,-951.284032,0.119705,10048895,22.9,1
2,cpu,1,3,-951.334532,0.119766,10058994,21.8,1
3,cpu,1,4,-951.467821,0.120067,10089040,21.8,0.282
4,cpu,1.5,1,-951.238531,0.119668,8512771,19.4,1
5,cpu,1.5,2,-951.528820,0.120280,8561078,19.6,1
6,cpu,1.5,3,-951.441966,0.119831,8552678,19.6,0.765
7,cpu,1.5,4,-951.382040,0.119987,8542135,19.9,0.0164
8,cpu,2,1,-951.408336,0.119877,7825455,19.8,1
9,cpu,2,2,-951.334334,0.119820,7813029,19.5,1
